# Module 16 Lab — LangGraph: State Graph Agent Workflows

In this lab you will build a **research pipeline** in LangGraph. The pipeline:

1. **Planner** — decomposes a question into sub-questions
2. **Researcher** — answers each sub-question (parallel with Send API in the final section)
3. **Synthesiser** — combines results into a coherent answer
4. **Critic** — reviews the synthesis and loops back if revision is needed

Topics covered:
- `TypedDict` state design
- Node functions and partial state updates
- Graph assembly: `add_node`, `add_edge`, `add_conditional_edges`
- `MemorySaver` checkpointing and `interrupt_before`
- Send API for dynamic fan-out

In [ ]:
!pip install langgraph langchain-anthropic langchain-openai

In [ ]:
import os
from langchain_anthropic import ChatAnthropic
# Alternatively, use OpenAI:
# from langchain_openai import ChatOpenAI

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
if not ANTHROPIC_API_KEY:
    raise ValueError("Set ANTHROPIC_API_KEY in Colab Secrets or environment")

MODEL_NAME = "claude-haiku-4-5-20251001"
llm = ChatAnthropic(model=MODEL_NAME, api_key=ANTHROPIC_API_KEY)

# OpenAI alternative:
# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
# llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY)

print("LLM ready:", MODEL_NAME)

---
## Section 1: TypedDict State

In LangGraph, **state is the single source of truth**. Every node:
- Receives the full current state as input
- Returns a **partial dict** — only the fields it updates

LangGraph merges the partial dict back into state automatically.

Design your state to hold everything any node might need to read or write. For list fields that accumulate results across nodes (like `research_results`), you can use `Annotated[list, operator.add]` to make LangGraph **append** rather than replace — but for this lab we handle accumulation manually to keep things explicit.

In [ ]:
from typing import TypedDict

class ResearchState(TypedDict):
    question: str           # The user's original question
    sub_questions: list     # Decomposed sub-questions from the planner
    research_results: list  # One result string per sub-question
    synthesis: str          # Combined answer from the synthesiser
    critique: str           # Reviewer feedback from the critic
    retry_count: int        # How many revision cycles have run

# Verify the TypedDict is importable and well-formed
sample_state: ResearchState = {
    "question": "How does LangGraph handle state?",
    "sub_questions": [],
    "research_results": [],
    "synthesis": "",
    "critique": "",
    "retry_count": 0
}
print("State fields:", list(sample_state.keys()))

---
## Section 2: Node Functions

Each node is a plain Python function: `(state: ResearchState) -> dict`.

The returned dict contains **only the fields this node changes** — LangGraph merges it into the full state. You never return the whole state object.

Implement all four nodes below.

### TODO 1 — Planner node

Call the LLM to decompose `state["question"]` into exactly 3 sub-questions.  
Return `{"sub_questions": [...]}`.  

Hint: prompt the model to output only a numbered list, then parse the lines.

In [ ]:
# TODO 1: Implement planner_node
def planner_node(state: ResearchState) -> dict:
    """Decompose the question into 3 sub-questions."""
    # YOUR CODE HERE
    pass

# === SOLUTION (reveal only if stuck) ===
# def planner_node(state: ResearchState) -> dict:
#     prompt = (
#         f"Break the following research question into exactly 3 focused sub-questions.\n"
#         f"Output only the sub-questions, one per line, numbered 1. 2. 3.\n\n"
#         f"Question: {state['question']}"
#     )
#     response = llm.invoke(prompt)
#     lines = [line.strip() for line in response.content.strip().splitlines() if line.strip()]
#     # Strip leading "1. ", "2. ", "3. " prefixes
#     sub_questions = []
#     for line in lines[:3]:
#         parts = line.split(".", 1)
#         sub_questions.append(parts[1].strip() if len(parts) > 1 else line)
#     return {"sub_questions": sub_questions}

### TODO 2 — Researcher node

Pick the next unanswered sub-question (index = `len(state["research_results"])`), simulate a search result, and return `{"research_results": state["research_results"] + [result]}`.

Keep it simple — the "search" can just be an LLM call that answers the sub-question in 2–3 sentences.

In [ ]:
# TODO 2: Implement researcher_node
def researcher_node(state: ResearchState) -> dict:
    """Answer the next unanswered sub-question."""
    # YOUR CODE HERE
    pass

# === SOLUTION (reveal only if stuck) ===
# def researcher_node(state: ResearchState) -> dict:
#     idx = len(state["research_results"])
#     sub_q = state["sub_questions"][idx]
#     prompt = f"Answer this research question in 2-3 sentences: {sub_q}"
#     response = llm.invoke(prompt)
#     result = f"[Sub-question {idx+1}] {sub_q}\nAnswer: {response.content.strip()}"
#     return {"research_results": state["research_results"] + [result]}

### TODO 3 — Synthesiser node

Join all `state["research_results"]` into a single coherent answer.  
Return `{"synthesis": ...}`.

In [ ]:
# TODO 3: Implement synthesiser_node
def synthesiser_node(state: ResearchState) -> dict:
    """Combine research results into a coherent synthesis."""
    # YOUR CODE HERE
    pass

# === SOLUTION (reveal only if stuck) ===
# def synthesiser_node(state: ResearchState) -> dict:
#     results_text = "\n\n".join(state["research_results"])
#     prompt = (
#         f"You have gathered the following research results:\n\n{results_text}\n\n"
#         f"Write a clear, concise synthesis (3-5 sentences) that answers the original question: "
#         f"{state['question']}"
#     )
#     response = llm.invoke(prompt)
#     return {"synthesis": response.content.strip()}

### TODO 4 — Critic node

Review `state["synthesis"]` and return a critique.  
If the synthesis is weak or incomplete, include the literal string `"NEEDS_REVISION"` in the critique.  
Return `{"critique": ..., "retry_count": state["retry_count"] + 1}`.

In [ ]:
# TODO 4: Implement critic_node
def critic_node(state: ResearchState) -> dict:
    """Critique the synthesis. Include 'NEEDS_REVISION' if it needs improvement."""
    # YOUR CODE HERE
    pass

# === SOLUTION (reveal only if stuck) ===
# def critic_node(state: ResearchState) -> dict:
#     prompt = (
#         f"Review this synthesis for quality, accuracy, and completeness:\n\n"
#         f"{state['synthesis']}\n\n"
#         f"If it needs improvement, begin your critique with 'NEEDS_REVISION: ' and explain why.\n"
#         f"If it is satisfactory, begin with 'APPROVED: ' and explain why."
#     )
#     response = llm.invoke(prompt)
#     critique = response.content.strip()
#     return {"critique": critique, "retry_count": state["retry_count"] + 1}

---
## Section 3: Graph Assembly

LangGraph graphs are built with `StateGraph`:

```python
graph = StateGraph(ResearchState)
graph.add_node("name", node_fn)
graph.add_edge("a", "b")          # unconditional edge
graph.add_conditional_edges("a", routing_fn, {"label": "node_name", ...})
graph.set_entry_point("first_node")
app = graph.compile()
```

The basic pipeline runs: `planner → researcher (×3) → synthesiser → critic`.

We loop the researcher three times (once per sub-question) using a simple routing function before wiring the conditional critic loop.

In [ ]:
from langgraph.graph import StateGraph, END

def route_after_research(state: ResearchState) -> str:
    """Loop back to researcher until all sub-questions are answered."""
    if len(state["research_results"]) < len(state["sub_questions"]):
        return "researcher"
    return "synthesiser"


graph_builder = StateGraph(ResearchState)

graph_builder.add_node("planner", planner_node)
graph_builder.add_node("researcher", researcher_node)
graph_builder.add_node("synthesiser", synthesiser_node)
graph_builder.add_node("critic", critic_node)

graph_builder.set_entry_point("planner")
graph_builder.add_edge("planner", "researcher")
graph_builder.add_conditional_edges(
    "researcher",
    route_after_research,
    {"researcher": "researcher", "synthesiser": "synthesiser"}
)
graph_builder.add_edge("synthesiser", "critic")

print("Graph nodes defined. Add the critic routing edge in TODO 5.")

### TODO 5 — Conditional edge after critic

Write `route_after_critic(state: ResearchState) -> str` that:
- Returns `"synthesiser"` if `"NEEDS_REVISION"` is in `state["critique"]` AND `state["retry_count"] < 2`
- Returns `END` otherwise

Then add it as a conditional edge from `"critic"`.

In [ ]:
# TODO 5: Write route_after_critic and add the conditional edge

def route_after_critic(state: ResearchState) -> str:
    # YOUR CODE HERE
    pass

# graph_builder.add_conditional_edges(
#     "critic",
#     route_after_critic,
#     {YOUR MAPPING HERE}
# )

# === SOLUTION (reveal only if stuck) ===
# def route_after_critic(state: ResearchState) -> str:
#     if "NEEDS_REVISION" in state["critique"] and state["retry_count"] < 2:
#         return "synthesiser"
#     return END
#
# graph_builder.add_conditional_edges(
#     "critic",
#     route_after_critic,
#     {"synthesiser": "synthesiser", END: END}
# )

# Compile (replace this with the version that includes your conditional edge)
app = graph_builder.compile()
print("Graph compiled.")

In [ ]:
# Run the basic pipeline
initial_state: ResearchState = {
    "question": "How does LangGraph handle state persistence across agent steps?",
    "sub_questions": [],
    "research_results": [],
    "synthesis": "",
    "critique": "",
    "retry_count": 0
}

result = app.invoke(initial_state)

print("=== Sub-questions ===")
for i, q in enumerate(result["sub_questions"], 1):
    print(f"  {i}. {q}")

print("\n=== Synthesis ===")
print(result["synthesis"])

print("\n=== Critique ===")
print(result["critique"])

print(f"\nRetry count: {result['retry_count']}")

---
## Section 4: Checkpointing

`MemorySaver` stores a snapshot of state after every node. This enables:
- **Pause and resume** — run to a breakpoint, inspect state, continue
- **Time-travel debugging** — retrieve any past state snapshot
- **Human-in-the-loop** — `interrupt_before=["node"]` pauses before that node runs

A `config` dict with `{"configurable": {"thread_id": "<unique-id>"}}` identifies the conversation thread.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

# Rebuild the graph with the checkpointer attached
graph_with_memory = StateGraph(ResearchState)
graph_with_memory.add_node("planner", planner_node)
graph_with_memory.add_node("researcher", researcher_node)
graph_with_memory.add_node("synthesiser", synthesiser_node)
graph_with_memory.add_node("critic", critic_node)
graph_with_memory.set_entry_point("planner")
graph_with_memory.add_edge("planner", "researcher")
graph_with_memory.add_conditional_edges(
    "researcher", route_after_research,
    {"researcher": "researcher", "synthesiser": "synthesiser"}
)
graph_with_memory.add_edge("synthesiser", "critic")
# Add your route_after_critic edge here too if TODO 5 is complete

app_mem = graph_with_memory.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "research-001"}}

result_mem = app_mem.invoke(
    {
        "question": "What are the key advantages of checkpointing in LangGraph?",
        "sub_questions": [],
        "research_results": [],
        "synthesis": "",
        "critique": "",
        "retry_count": 0
    },
    config=config
)

print("Synthesis:", result_mem["synthesis"])

### TODO 6 — Interrupt before critic and inspect state

Recompile `app_mem` with `interrupt_before=["critic"]`.  
Run `app_mem.invoke(...)` — it will pause before the critic node.  
Then call `app_mem.get_state(config)` to inspect the paused state.  
Finally, resume by calling `app_mem.invoke(None, config=config)`.

In [ ]:
# TODO 6: interrupt_before=["critic"] and inspect mid-run state

# YOUR CODE HERE
# 1. Recompile with interrupt_before=["critic"]
# 2. app_mem_paused.invoke(initial_state, config=config2)
# 3. snapshot = app_mem_paused.get_state(config2)  — print snapshot.values
# 4. app_mem_paused.invoke(None, config=config2)   — resume

# === SOLUTION (reveal only if stuck) ===
# memory2 = MemorySaver()
# app_mem_paused = graph_with_memory.compile(
#     checkpointer=memory2,
#     interrupt_before=["critic"]
# )
# config2 = {"configurable": {"thread_id": "research-002"}}
#
# # Run until the interrupt
# app_mem_paused.invoke(
#     {"question": "Why is state persistence important for multi-step agents?",
#      "sub_questions": [], "research_results": [],
#      "synthesis": "", "critique": "", "retry_count": 0},
#     config=config2
# )
#
# # Inspect state at the interrupt point
# snapshot = app_mem_paused.get_state(config2)
# print("State at interrupt:")
# print("  synthesis:", snapshot.values["synthesis"][:120], "...")
# print("  next nodes:", snapshot.next)
#
# # Resume
# final = app_mem_paused.invoke(None, config=config2)
# print("\nFinal critique:", final["critique"])

---
## Section 5: Send API — Dynamic Fan-out

The sequential researcher loop (Section 3) answers sub-questions one at a time. The **Send API** lets you dispatch multiple nodes in parallel — each with its own state slice.

```python
from langgraph.types import Send

def fan_out(state: ResearchState):
    return [Send("researcher", {"question": q}) for q in state["sub_questions"]]
```

This returns a **list of `Send` objects** from a conditional edge function. LangGraph launches all researcher nodes simultaneously and merges their outputs (you need `Annotated[list, operator.add]` on `research_results` for merge to work correctly).

### Annotated state for parallel merging

When multiple nodes write to the same list field simultaneously, LangGraph needs to know how to merge them. Use `Annotated` with `operator.add`:

In [ ]:
import operator
from typing import Annotated
from langgraph.types import Send

# New state definition with Annotated list for parallel merge
class ParallelResearchState(TypedDict):
    question: str
    sub_questions: list
    research_results: Annotated[list, operator.add]  # append-merge on parallel writes
    synthesis: str
    critique: str
    retry_count: int

print("ParallelResearchState defined with Annotated[list, operator.add] for research_results")

### TODO 7 — Fan-out researcher with Send API

Write `fan_out_researcher(state: ParallelResearchState)` that returns a list of `Send` objects — one per sub-question.

Each `Send` dispatches `"parallel_researcher"` with `{"question": q, ...}`. The researcher node will receive only that slice of state.

Then wire it as a conditional edge from `"planner"` in a new graph.

In [ ]:
# TODO 7: Implement fan_out_researcher and wire it into a parallel graph

def parallel_researcher_node(state: dict) -> dict:
    """Answers a single sub-question. Receives state slice with 'question' key."""
    sub_q = state.get("question", "")
    prompt = f"Answer this research question in 2-3 sentences: {sub_q}"
    response = llm.invoke(prompt)
    result = f"[Sub-question] {sub_q}\nAnswer: {response.content.strip()}"
    return {"research_results": [result]}


def fan_out_researcher(state: ParallelResearchState):
    """Return a Send for each sub-question to run researchers in parallel."""
    # TODO: return [Send("parallel_researcher", {"question": q}) for q in state["sub_questions"]]
    # YOUR CODE HERE
    pass


# Wire the parallel graph
# YOUR CODE HERE
# parallel_graph = StateGraph(ParallelResearchState)
# parallel_graph.add_node("planner", planner_node)
# parallel_graph.add_node("parallel_researcher", parallel_researcher_node)
# parallel_graph.add_node("synthesiser", synthesiser_node)
# parallel_graph.set_entry_point("planner")
# parallel_graph.add_conditional_edges("planner", fan_out_researcher, ["parallel_researcher"])
# parallel_graph.add_edge("parallel_researcher", "synthesiser")
# parallel_graph.add_edge("synthesiser", END)
# parallel_app = parallel_graph.compile()

# === SOLUTION (reveal only if stuck) ===
# def fan_out_researcher(state: ParallelResearchState):
#     return [Send("parallel_researcher", {"question": q}) for q in state["sub_questions"]]
#
# parallel_graph = StateGraph(ParallelResearchState)
# parallel_graph.add_node("planner", planner_node)
# parallel_graph.add_node("parallel_researcher", parallel_researcher_node)
# parallel_graph.add_node("synthesiser", synthesiser_node)
# parallel_graph.set_entry_point("planner")
# parallel_graph.add_conditional_edges(
#     "planner", fan_out_researcher, ["parallel_researcher"]
# )
# parallel_graph.add_edge("parallel_researcher", "synthesiser")
# parallel_graph.add_edge("synthesiser", END)
# parallel_app = parallel_graph.compile()
#
# parallel_result = parallel_app.invoke({
#     "question": "What is LangGraph and how does it relate to LangChain?",
#     "sub_questions": [],
#     "research_results": [],
#     "synthesis": "",
#     "critique": "",
#     "retry_count": 0
# })
# print("Parallel synthesis:", parallel_result["synthesis"])

---
## Wrap-up

You have built a full LangGraph research pipeline. Key takeaways:

- **State** is a `TypedDict` — the single source of truth that all nodes read from and write partial updates to.
- **Nodes** return partial dicts — only the fields they change.
- **Conditional edges** enable routing logic (critic loop, fan-out)
- **MemorySaver + interrupt_before** enable human-in-the-loop workflows and time-travel debugging.
- **Send API** enables dynamic parallel fan-out — one `Send` per sub-task, results merged via `Annotated[list, operator.add]`.

Next steps:
- Replace the simulated LLM calls with real tool use (Tavily, SerpAPI)
- Swap `MemorySaver` for `SqliteSaver` or `PostgresSaver` for persistent cross-session state
- Add a human review step with `interrupt_before` between synthesiser and critic